# 03. AI Search 인덱싱 파이프라인 실행

이 노트북에서는:
1. AI Search 인덱스 스키마 및 설계 원칙을 이해합니다.
2. Blob Storage에 있는 크롤링 데이터 중 어떤 파일들이 인덱싱될 대상인지 확인합니다.
3. AI Search 인덱싱 파이프라인(skillset, datasource, indexer)을 생성하고 실행합니다.

권장 순서: `02 → 03 → 04`

## 1. 환경 설정

In [1]:
import os, sys
from dotenv import load_dotenv

sys.path.insert(0, os.path.abspath('..'))
load_dotenv('../.env')

SEARCH_ENDPOINT = os.environ['AZURE_SEARCH_SERVICE_ENDPOINT']
OPENAI_ENDPOINT = os.environ['AZURE_OPENAI_ENDPOINT']
STORAGE_NAME = os.environ.get('AZURE_STORAGE_ACCOUNT_NAME', '')
STORAGE_CONTAINER = os.environ.get('AZURE_STORAGE_CONTAINER_NAME', 'raw-documents')

print(f"Search Endpoint: {SEARCH_ENDPOINT}")
print(f"OpenAI Endpoint: {OPENAI_ENDPOINT}")
print(f"Storage         : {STORAGE_NAME}/{STORAGE_CONTAINER}")

Search Endpoint: https://search-ragi-63325wdo.search.windows.net
OpenAI Endpoint: https://ais-ragi-63325wdo.cognitiveservices.azure.com/
Storage         : stragi63325wdoby/raw-documents


## 2. 인덱스 스키마 설계 — 법률 4개 인덱스

법률 데이터 특성에 맞게 인덱스별로 스키마를 분리 설계합니다.  
모든 인덱스는 **하이브리드 검색(키워드 + 벡터) + Semantic Ranker** 구조를 공유합니다.

### 공통 설계 원칙

| 구분 | 타입 | 설정 | 용도 |
|------|------|------|------|
| **Key 필드** | SimpleField | `key=True, filterable=True` | 문서 고유 ID |
| **메타데이터** | SearchableField | `filterable=True, sortable=True` | 날짜 범위 필터, 정렬 |
| **열거형 메타** | SearchableField | `filterable=True, facetable=True` | 심급·유형 패싯 UI |
| **다중값 메타** | SearchableField | `type=Collection(String), filterable=True` | 관련법령·주제어 정확 필터 |
| **본문 텍스트** | SearchableField | `analyzer_name="ko.microsoft"` | 한국어 키워드 검색 |
| **벡터 임베딩** | SearchField | `searchable=True, hidden=True` | 의미 기반 벡터 검색 |

**주요 결정**:
- **임베딩 비용 절약**: 전체 문서가 아닌 핵심 필드만 임베딩 (판결요지/결정요지/회답/재결요지)
- **`Collection(String)` 필터**: "민법,형법" 단일 텍스트로 저장 불가 → 배열로 변경
- **`hidden=True`**: 3072 float 벡터를 응답에서 제외 (네트워크 비용 감소)

In [2]:
from src.search.legal_indexes import PREC_INDEX, CONST_INDEX, INTERP_INDEX, ADMIN_INDEX

# 4개 인덱스 정보
INDEX_META = {
    PREC_INDEX: {
        "name": "판례",
        "source": "prec",
        "blob_prefix": "prec/",
        "embedding_fields": ["판결요지"],
        "semantic_config": "prec-semantic",
    },
    CONST_INDEX: {
        "name": "헌법재판소 결정례",
        "source": "detc",
        "blob_prefix": "detc/",
        "embedding_fields": ["결정요지"],
        "semantic_config": "const-semantic",
    },
    INTERP_INDEX: {
        "name": "법제처 해석례",
        "source": "expc",
        "blob_prefix": "expc/",
        "embedding_fields": ["회답"],
        "semantic_config": "interp-semantic",
    },
    ADMIN_INDEX: {
        "name": "행정심판 재결례",
        "source": "admrul",
        "blob_prefix": "admrul/",
        "embedding_fields": ["재결요지"],
        "semantic_config": "admin-semantic",
    },
}

print("\n=== 4개 AI Search 인덱스 ===\n")
print(f"{'인덱스명':<30} {'한국어명':<20} {'Blob 경로':<15} {'임베딩 대상 필드'}")
print("-" * 85)
for idx_name, meta in INDEX_META.items():
    print(
        f"{idx_name:<30} "
        f"{meta['name']:<20} "
        f"{meta['blob_prefix']:<15} "
        f"{', '.join(meta['embedding_fields'])}"
    )


=== 4개 AI Search 인덱스 ===

인덱스명                           한국어명                 Blob 경로         임베딩 대상 필드
-------------------------------------------------------------------------------------
prec-court-index               판례                   prec/           판결요지
const-court-index              헌법재판소 결정례            detc/           결정요지
legis-interp-index             법제처 해석례              expc/           회답
admin-appeal-index             행정심판 재결례             admrul/         재결요지


## 3. 인덱싱될 Blob 파일 확인

Blob Storage에서 실제로 AI Search로 인덱싱될 4개 소스별 파일들을 확인합니다.

In [3]:
from azure.identity import DefaultAzureCredential
from azure.storage.blob import BlobServiceClient

credential = DefaultAzureCredential(
    exclude_managed_identity_credential=True,
    exclude_workload_identity_credential=True,
)
blob_svc = BlobServiceClient(
    account_url=f"https://{STORAGE_NAME}.blob.core.windows.net",
    credential=credential,
)
container_client = blob_svc.get_container_client(STORAGE_CONTAINER)

# 각 소스별 Blob 파일 집계
print("\n=== Blob 파일 현황 (인덱싱될 데이터) ===\n")
total_files = 0
for idx_name, meta in INDEX_META.items():
    prefix = meta['blob_prefix']
    blobs = [b for b in container_client.list_blobs(name_starts_with=prefix) if b.name.endswith('.json')]
    total_files += len(blobs)

    # 날짜별 집계
    dates = sorted(set(b.name.split('/')[1] for b in blobs if b.name.count('/') >= 2), reverse=True)

    print(f"[{meta['name']}] ({prefix})")
    print(f"  전체: {len(blobs)}개 파일, {len(dates)}개 날짜")
    for d in dates:
        cnt = sum(1 for b in blobs if f"{prefix}{d}/" in b.name)
        print(f"    {d}: {cnt}개")
    print()

print(f"✅ 인덱싱될 총 파일: {total_files}개\n")



=== Blob 파일 현황 (인덱싱될 데이터) ===

[판례] (prec/)
  전체: 105941개 파일, 6개 날짜
    2026-05-19: 34043개
    2026-05-18: 52121개
    2026-05-17: 760개
    2026-05-16: 176개
    2026-05-13: 9573개
    2026-05-12: 9268개

[헌법재판소 결정례] (detc/)
  전체: 38093개 파일, 6개 날짜
    2026-05-19: 65개
    2026-05-18: 12673개
    2026-05-17: 239개
    2026-05-16: 131개
    2026-05-13: 12004개
    2026-05-12: 12981개

[법제처 해석례] (expc/)
  전체: 8715개 파일, 1개 날짜
    2026-05-12: 8715개

[행정심판 재결례] (admrul/)
  전체: 29108개 파일, 6개 날짜
    2026-05-19: 3633개
    2026-05-18: 16827개
    2026-05-17: 886개
    2026-05-16: 931개
    2026-05-13: 3282개
    2026-05-12: 3549개

✅ 인덱싱될 총 파일: 181857개



In [13]:
# 전처리된 JSONL 파일 현황 + Raw Blob 대비 비교
PROC_CONTAINER = os.environ.get('AZURE_SEARCH_INDEXING_CONTAINER', 'processed-documents')
proc_client = blob_svc.get_container_client(PROC_CONTAINER)

print(f"\n=== Processed JSONL 현황 ({PROC_CONTAINER}) vs Raw Blob ({STORAGE_CONTAINER}) ===\n")
print(f"{'소스':<20} {'Raw .json':>10} {'JSONL 파일':>12} {'JSONL docs':>12}  비고")
print("-" * 75)

total_raw = 0
total_jsonl_files = 0
total_jsonl_docs = 0

for idx_name, meta in INDEX_META.items():
    prefix = meta['blob_prefix']
    source = meta['source']

    # Raw blob count (이전 셀에서 계산한 것과 동일)
    raw_blobs = [b for b in container_client.list_blobs(name_starts_with=prefix) if b.name.endswith('.json')]
    raw_n = len(raw_blobs)

    # Processed JSONL 파일 수 + 각 파일 내 문서 수 (줄 수 = 문서 수)
    jsonl_blobs = sorted(
        [b for b in proc_client.list_blobs(name_starts_with=prefix) if b.name.endswith('.jsonl')],
        key=lambda b: b.name,
    )
    jsonl_n = len(jsonl_blobs)
    jsonl_docs = 0
    for jb in jsonl_blobs:
        data = proc_client.get_blob_client(jb.name).download_blob().readall()
        lines = [l for l in data.decode('utf-8').splitlines() if l.strip()]
        jsonl_docs += len(lines)

    total_raw += raw_n
    total_jsonl_files += jsonl_n
    total_jsonl_docs += jsonl_docs

    note = "✅" if jsonl_docs > 0 else "⚠️ JSONL 없음"
    print(f"[{meta['name']}] ({prefix:<10}) {raw_n:>10,} {jsonl_n:>12,} {jsonl_docs:>12,}  {note}")

print("-" * 75)
print(f"{'합계':<20} {total_raw:>10,} {total_jsonl_files:>12,} {total_jsonl_docs:>12,}")
print()

# 비교 요약
if total_jsonl_docs == 0:
    print("⚠️  전처리된 JSONL 이 아직 없습니다. 02-data-crawling 노트북에서 크롤+전처리를 먼저 실행하세요.")
elif total_jsonl_docs == total_raw:
    print(f"✅ Raw JSON ({total_raw:,}건) = JSONL docs ({total_jsonl_docs:,}건) — 1:1 매핑 정상")
else:
    diff = total_jsonl_docs - total_raw
    print(f"ℹ️  Raw JSON: {total_raw:,}건 → JSONL docs: {total_jsonl_docs:,}건 (차이: {diff:+,})")
    if diff > 0:
        print("   JSONL에 더 많음 — 여러 크롤 날짜의 파일이 하나의 JSONL로 병합된 경우")
    else:
        print("   JSONL에 더 적음 — 아직 전처리되지 않은 파일이 있을 수 있음")


=== Processed JSONL 현황 (processed-documents) vs Raw Blob (raw-documents) ===

소스                    Raw .json     JSONL 파일   JSONL docs  비고
---------------------------------------------------------------------------


[판례] (prec/     )    105,941           13      105,906  ✅
[헌법재판소 결정례] (detc/     )     38,093            7       38,093  ✅
[법제처 해석례] (expc/     )      8,715            1        8,715  ✅
[행정심판 재결례] (admrul/   )     29,108            8       29,108  ✅
---------------------------------------------------------------------------
합계                      181,857           29      181,822

ℹ️  Raw JSON: 181,857건 → JSONL docs: 181,822건 (차이: -35)
   JSONL에 더 적음 — 아직 전처리되지 않은 파일이 있을 수 있음


## 4. AI Search 인덱싱 파이프라인 실행 (소스별 분리)

각 소스별로 인덱서를 개별 실행합니다. 한 소스가 실패해도 다른 소스에는 영향이 없으며, 시간/실패 건수도 분리해서 확인할 수 있습니다.

각 셀이 수행하는 작업:
1. 해당 소스 인덱스 생성 (스키마, Semantic Ranker 포함)
2. Skillset / DataSource / Indexer 생성
3. Indexer 즉시 실행 + 완료 대기 (최대 15분)


In [ ]:
import subprocess
import time
from datetime import datetime


def run_indexing(source: str, label: str):
    """단일 소스 인덱싱 파이프라인 실행 (생성 + Indexer 즉시 실행 + 완료 대기)"""
    print("\n" + "=" * 80)
    print(f"[{label}] AI Search 인덱싱 시작 (source={source})")
    print("=" * 80)
    print(f"시작 시간: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 80 + "\n")

    start = time.time()
    result = subprocess.run(
        [sys.executable, "scripts/setup_ai_search_pipeline.py", "--source", source, "--run"],
        cwd=os.path.abspath(".."),
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace",
    )
    elapsed = time.time() - start

    print(result.stdout)
    if result.stderr:
        print("\n[stderr]")
        print(result.stderr)

    print("\n" + "=" * 80)
    print(f"[{label}] 인덱싱 완료")
    print(f"종료 시간: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"소요 시간: {elapsed:.1f}초 ({int(elapsed // 60)}분 {int(elapsed % 60)}초)")
    print(f"종료 코드: {result.returncode} ({'✅ 성공' if result.returncode == 0 else '❌ 실패'})")
    print("=" * 80)
    return result.returncode, elapsed


In [6]:
# (1/4) 판례 인덱싱
#run_indexing("prec", "판례")


In [7]:
# (2/4) 헌법재판소 결정례 인덱싱
#run_indexing("detc", "헌법재판소 결정례")


In [8]:
# (3/4) 법제처 해석례 인덱싱
#run_indexing("expc", "법제처 해석례")


In [9]:
# (4/4) 행정심판 재결례 인덱싱
#run_indexing("admrul", "행정심판 재결례")


## 5. 인덱스 통계 확인

인덱싱 완료 후 4개 인덱스에 저장된 문서 개수를 확인합니다.

In [10]:
from src.search.legal_indexes import LegalIndexManager

mgr = LegalIndexManager(
    endpoint=SEARCH_ENDPOINT,
    admin_key=os.environ.get('AZURE_SEARCH_ADMIN_KEY'),
)

print("\n=== 인덱스 문서 개수 ===\n")
stats = mgr.get_all_stats()
total = 0
for s in stats:
    idx_name = s['index_name']
    meta = INDEX_META.get(idx_name, {})
    korean_name = meta.get('name', idx_name)
    count = s['document_count']
    total += count
    status = "✅" if count > 0 else "⚠️"
    print(f"{status} {korean_name:<25} ({idx_name:<30}): {count:,}건")

print(f"\n총합: {total:,}건")

if total > 0:
    print("\n✅ 인덱싱 완료!")
else:
    print("\n⚠️  아직 인덱싱된 문서가 없습니다. Blob 파일을 확인하세요.")


=== 인덱스 문서 개수 ===



✅ 판례                        (prec-court-index              ): 105,906건
✅ 헌법재판소 결정례                 (const-court-index             ): 38,086건
✅ 법제처 해석례                   (legis-interp-index            ): 8,715건
✅ 행정심판 재결례                  (admin-appeal-index            ): 29,107건

총합: 181,814건

✅ 인덱싱 완료!


In [12]:
import os
from datetime import datetime
import pandas as pd
import requests
from azure.identity import DefaultAzureCredential
from azure.storage.blob import BlobServiceClient
from src.search.legal_indexes import LegalIndexManager

# 인덱스별 Blob 파일 수 / 인덱싱 결과(문서수, 소요시간) 비교 리포트 생성


# 1) Blob 파일 수 집계
if "container_client" not in globals():
    credential = DefaultAzureCredential(
        exclude_managed_identity_credential=True,
        exclude_workload_identity_credential=True,
    )
    blob_svc = BlobServiceClient(
        account_url=f"https://{STORAGE_NAME}.blob.core.windows.net",
        credential=credential,
    )
    container_client = blob_svc.get_container_client(STORAGE_CONTAINER)

blob_counts = {}
for idx_name, meta in INDEX_META.items():
    prefix = meta["blob_prefix"]
    blob_counts[idx_name] = sum(
        1 for b in container_client.list_blobs(name_starts_with=prefix)
        if b.name.endswith(".json")
    )

# 2) 인덱스 문서 수 집계
if "mgr" not in globals():
    mgr = LegalIndexManager(
        endpoint=SEARCH_ENDPOINT,
        admin_key=os.environ.get("AZURE_SEARCH_ADMIN_KEY"),
    )

stats = mgr.get_all_stats()
doc_count_map = {s["index_name"]: s["document_count"] for s in stats}

# 3) Indexer 상태 조회 (소요시간/처리건수)
api_version = "2024-07-01"
admin_key = os.environ.get("AZURE_SEARCH_ADMIN_KEY")

if admin_key:
    headers = {"api-key": admin_key, "Content-Type": "application/json"}
else:
    cred = DefaultAzureCredential(
        exclude_managed_identity_credential=True,
        exclude_workload_identity_credential=True,
    )
    token = cred.get_token("https://search.azure.com/.default").token
    headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}

indexers_url = f"{SEARCH_ENDPOINT}/indexers?api-version={api_version}"
indexers_resp = requests.get(indexers_url, headers=headers, timeout=30)
indexers_resp.raise_for_status()
all_indexers = indexers_resp.json().get("value", [])

# targetIndexName 기준으로 인덱스에 연결된 인덱서 찾기
idx_to_indexers = {k: [] for k in INDEX_META.keys()}
for ixr in all_indexers:
    target_idx = ixr.get("targetIndexName")
    if target_idx in idx_to_indexers:
        idx_to_indexers[target_idx].append(ixr.get("name"))

def _sec(start, end):
    if not start or not end:
        return None
    s = datetime.fromisoformat(start.replace("Z", "+00:00"))
    e = datetime.fromisoformat(end.replace("Z", "+00:00"))
    return (e - s).total_seconds()

status_map = {}
for idx_name, indexer_names in idx_to_indexers.items():
    best = None
    best_start = None

    for name in indexer_names:
        st_url = f"{SEARCH_ENDPOINT}/indexers('{name}')/status?api-version={api_version}"
        st_resp = requests.get(st_url, headers=headers, timeout=30)
        if st_resp.status_code >= 400:
            continue
        data = st_resp.json()
        last = data.get("lastResult", {}) or {}

        start_t = last.get("startTime")
        if start_t and (best_start is None or start_t > best_start):
            best_start = start_t
            best = {
                "indexer_name": name,
                "status": last.get("status"),
                "items_processed": last.get("itemCount"),
                "items_failed": last.get("failedItemCount"),
                "elapsed_sec": _sec(last.get("startTime"), last.get("endTime")),
                "start_time": last.get("startTime"),
                "end_time": last.get("endTime"),
            }

    status_map[idx_name] = best or {
        "indexer_name": None,
        "status": None,
        "items_processed": None,
        "items_failed": None,
        "elapsed_sec": None,
        "start_time": None,
        "end_time": None,
    }

# 4) 통합 표 생성
rows = []
for idx_name, meta in INDEX_META.items():
    blob_n = blob_counts.get(idx_name, 0)
    doc_n = doc_count_map.get(idx_name, 0)
    st = status_map.get(idx_name, {})

    rows.append({
        "인덱스명": idx_name,
        "한국어명": meta["name"],
        "Blob JSON 파일수": blob_n,
        "인덱스 문서수": doc_n,
        "차이(문서-파일)": doc_n - blob_n,
        "Indexer": st.get("indexer_name"),
        "Indexer 상태": st.get("status"),
        "처리건수(itemCount)": st.get("items_processed"),
        "실패건수(failedItemCount)": st.get("items_failed"),
        "소요시간(초)": st.get("elapsed_sec"),
    })

df_report = pd.DataFrame(rows)
display(df_report)

total_blob = int(df_report["Blob JSON 파일수"].sum())
total_docs = int(df_report["인덱스 문서수"].sum())
total_gap = total_docs - total_blob

# 5) REPORT.md 생성
md_lines = []
md_lines.append("# 인덱싱 리포트")
md_lines.append("")
md_lines.append(f"- 생성 시각: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
md_lines.append(f"- Storage Container: `{STORAGE_NAME}/{STORAGE_CONTAINER}`")
md_lines.append("")
md_lines.append("## 인덱스별 요약")
md_lines.append("")
md_lines.append(df_report.to_markdown(index=False))
md_lines.append("")
md_lines.append("## 총계 비교")
md_lines.append("")
md_lines.append(f"- Blob JSON 총 파일수: **{total_blob:,}**")
md_lines.append(f"- 인덱스 총 문서수: **{total_docs:,}**")
md_lines.append(f"- 차이(문서-파일): **{total_gap:+,}**")
md_lines.append("")
if total_gap == 0:
    md_lines.append("✅ Blob 파일 수와 인덱스 문서 수가 일치합니다.")
elif total_gap > 0:
    md_lines.append("ℹ️ 인덱스 문서 수가 더 많습니다. (파일 1개당 다수 문서 분할 인덱싱 가능)")
else:
    md_lines.append("⚠️ 인덱스 문서 수가 더 적습니다. 인덱싱 실패/누락 여부 점검이 필요합니다.")

with open("REPORT.md", "w", encoding="utf-8") as f:
    f.write("\n".join(md_lines))

print(f"\nREPORT.md 생성 완료")
print(f"- Blob 총 파일수: {total_blob:,}")
print(f"- 인덱스 총 문서수: {total_docs:,}")
print(f"- 차이: {total_gap:+,}")

,인덱스명,한국어명,Blob JSON 파일수,인덱스 문서수,차이(문서-파일),Indexer,Indexer 상태,처리건수(itemCount),실패건수(failedItemCount),소요시간(초)
0,prec-court-index,판례,105941,105906,-35,prec-blob-indexer,success,None,None,5753.744
1,const-court-index,헌법재판소 결정례,38093,38086,-7,const-blob-indexer,success,None,None,7.537
2,legis-interp-index,법제처 해석례,8715,8715,0,interp-blob-indexer,success,None,None,884.782
3,admin-appeal-index,행정심판 재결례,29108,29107,-1,admin-blob-indexer,success,None,None,3125.223



REPORT.md 생성 완료
- Blob 총 파일수: 181,857
- 인덱스 총 문서수: 181,814
- 차이: -43


## 6. Indexer Caching 효과 비교 실험 — Reindex vs Incremental Update

[scripts/setup_ai_search_pipeline.py](../scripts/setup_ai_search_pipeline.py) 의 `SETUP_ENABLE_CACHE=1` 옵션이 활성화되면 indexer 정의에 `cache` 블록이 추가되어 **incremental enrichment cache** 가 켜집니다.

### 용어 정리

| 용어 | 정의 | 어떻게 발생? |
|---|---|---|
| **Reindexing (전체 재인덱싱)** | Indexer `reset` 후 `run` → change-tracking 초기화 → **모든 blob 재처리** | `setup_ai_search_pipeline.py --run` 이 항상 reset 후 실행 |
| **Incremental Update (증분 갱신)** | `reset` 없이 `run` 만 → 변경된 blob (LastModified 기준) **만** 재처리 | REST `POST /indexers/{name}/run` 직접 호출 |

> **중요 포인트**: `reset` 은 *change tracking* 만 무효화하고 **enrichment cache 는 그대로 유지**됩니다.  
> 따라서 cache=ON 상태에서 reindex 를 다시 돌리면 → blob 은 전부 다시 다운로드/파싱되지만 **skill (embedding 등) 호출은 cache HIT 으로 건너뜁니다**. 임베딩 비용/시간 절감 효과를 reindex 때도 누릴 수 있습니다.

### 비교 시나리오 (실험 대상: `detc` — 변경분(신규 결정례)이 자주 추가되는 소스)

| | 시나리오 | 동작 | 종류 | cache 상태 | 예상 |
|---|---|---|---|---|---|
| **A** | 파이프라인 초기화 | 생성만 (`--run` 없음), `SETUP_ENABLE_CACHE=0` | Setup | — | Index/Skillset/DS/Indexer 생성 확인 |
| **B** | 캐시 ON reindex (1차) | `reset` + `run`, `SETUP_ENABLE_CACHE=1` | Reindex | 비어있음→채워짐 | baseline (전체 처리 + 캐시 채우기) |
| **B.5** | 신규 데이터 크롤링 | `crawl` 호출 → `raw-documents/detc/{today}/` 신규 N개 | Data ingest | — | C 의 변경분 생성 |
| **C** | 캐시 ON Incremental Update | `run` 만 (reset 없음) | Update | warm | 신규 N개만 처리, 기존은 skip → 매우 빠름 |
| **D** | 캐시 ON reindex (2차) — **캐시 재사용 검증** | `reset` + `run`, `SETUP_ENABLE_CACHE=1` 그대로 | Reindex | warm | blob 은 전부 재처리하지만 임베딩은 cache HIT → B 대비 단축 |

> D 의 핵심: blob 다운로드/파싱 시간은 거의 그대로지만, **임베딩 skill 호출 시간/비용**이 줄어듭니다. B 와 D 의 차이가 곧 "cache 가 reindex 때 절약해주는 양" 입니다.

In [ ]:
# 실험 헬퍼: setup 스크립트 호출 (env 로 캐시 토글) / indexer 직접 실행 (reset 없이)
import os, sys, time, subprocess, requests
from datetime import datetime
from azure.identity import DefaultAzureCredential

EXP_SOURCE = "detc"  # 변경분(신규 결정례)이 자주 추가되는 소스 → B.5 신규 데이터 생성에 유리
EXP_INDEXER = {"prec": "prec-blob-indexer", "detc": "const-blob-indexer",
               "expc": "interp-blob-indexer", "admrul": "admin-blob-indexer"}[EXP_SOURCE]
API_VERSION = "2024-07-01"

_cred_exp = DefaultAzureCredential(
    exclude_managed_identity_credential=True,
    exclude_workload_identity_credential=True,
)

def _exp_hdr():
    tok = _cred_exp.get_token("https://search.azure.com/.default").token
    return {"Authorization": f"Bearer {tok}", "Content-Type": "application/json"}

def _get_indexer_status(indexer_name: str) -> dict:
    st_url = f"{SEARCH_ENDPOINT}/indexers('{indexer_name}')/status?api-version={API_VERSION}"
    sr = requests.get(st_url, headers=_exp_hdr(), timeout=30)
    sr.raise_for_status()
    return sr.json()

def run_setup(source: str, cache_on: bool, run: bool = True) -> tuple[int, float, str]:
    """setup_ai_search_pipeline.py 호출. cache_on=True 면 SETUP_ENABLE_CACHE=1 주입.
    run=False 면 파이프라인 생성만 하고 indexer 실행은 하지 않음."""
    env = os.environ.copy()
    env["SETUP_ENABLE_CACHE"] = "1" if cache_on else "0"
    cmd = [sys.executable, "scripts/setup_ai_search_pipeline.py", "--source", source]
    if run:
        cmd.append("--run")
    t0 = time.time()
    result = subprocess.run(
        cmd,
        cwd=os.path.abspath(".."),
        capture_output=True, text=True, encoding="utf-8", errors="replace",
        env=env,
    )
    elapsed = time.time() - t0
    return result.returncode, elapsed, (result.stdout or "") + (result.stderr or "")

def run_indexer_only(indexer_name: str, poll_interval: int = 5) -> dict:
    """Indexer reset 없이 단순 run + 완료까지 폴링. itemCount / failedItemCount / 소요시간 반환.

    핵심: POST /run 직후 lastResult 는 이전 실행 결과 그대로이므로,
    (1) 호출 전 prev_start_time 을 캡처하고
    (2) lastResult.startTime 이 새 값으로 바뀌고 + status 가 terminal 상태일 때만 break.
    """
    # 0) 이전 run 의 startTime 캡처 (stale 결과 감지용)
    try:
        prev_last = _get_indexer_status(indexer_name).get("lastResult") or {}
        prev_start = prev_last.get("startTime")
    except Exception:
        prev_start = None

    # 1) run trigger
    run_url = f"{SEARCH_ENDPOINT}/indexers/{indexer_name}/run?api-version={API_VERSION}"
    r = requests.post(run_url, headers=_exp_hdr(), timeout=60)
    if r.status_code not in (202, 204):
        raise RuntimeError(f"run failed: {r.status_code} {r.text[:300]}")

    t0 = time.time()
    # 2) 새 run 이 registered 될 때까지 살짝 대기 + startTime 갱신 확인
    time.sleep(3)
    last = {}
    while True:
        data = _get_indexer_status(indexer_name)
        last = data.get("lastResult", {}) or {}
        st = last.get("status")
        cur_start = last.get("startTime")
        items  = last.get("itemCount")
        failed = last.get("failedItemCount")
        elapsed = int(time.time() - t0)

        # 새 run 이 lastResult 에 반영되었는가?
        is_new_run = (cur_start is not None) and (cur_start != prev_start)

        print(f"    [{elapsed:>4d}s] status={st}  startTime={cur_start}  items={items}  failed={failed}  "
              f"{'(new run)' if is_new_run else '(still prev)'}")

        if is_new_run and st in ("success", "transientFailure", "persistentFailure", "error"):
            break
        time.sleep(poll_interval)

    return {
        "status": st,
        "items_processed": items,
        "items_failed": failed,
        "elapsed_sec": round(time.time() - t0, 2),
        "start_time": last.get("startTime"),
        "end_time": last.get("endTime"),
        "prev_start_time": prev_start,
    }

print(f"실험 대상 source = {EXP_SOURCE}  /  indexer = {EXP_INDEXER}")
print(f"Search Endpoint   = {SEARCH_ENDPOINT}")

실험 대상 source = detc  /  indexer = const-blob-indexer
Search Endpoint   = https://search-ragi-63325wdo.search.windows.net


In [ ]:
# 시나리오 A: 캐시 OFF 로 파이프라인 초기화 (생성만, indexer 실행 없음)
print("="*70, f"\nA. SETUP_ENABLE_CACHE=0  →  파이프라인 초기화  (source={EXP_SOURCE})", "\n"+"="*70)
print(f"시작: {datetime.now().strftime('%H:%M:%S')}")

rc_a, elapsed_a, log_a = run_setup(EXP_SOURCE, cache_on=False, run=False)

# 마지막 30줄만 출력
tail = "\n".join(log_a.splitlines()[-30:])
print(tail)
print(f"\n[A 결과] rc={rc_a}  소요={elapsed_a:.1f}s ({'✅ 성공' if rc_a == 0 else '❌ 실패'})")
print("  → Index / Skillset / DataSource / Indexer 생성 완료 (실행은 B 에서)")

A. SETUP_ENABLE_CACHE=0  →  전체 재인덱싱  (source=detc) 
시작: 13:01:59
    상태: inProgress (처리 13000건, 실패 3건)
    상태: inProgress (처리 13000건, 실패 3건)
    상태: inProgress (처리 13000건, 실패 3건)
    상태: inProgress (처리 13500건, 실패 3건)
    상태: inProgress (처리 13500건, 실패 3건)
    상태: inProgress (처리 13500건, 실패 3건)
    상태: inProgress (처리 13500건, 실패 3건)
    상태: inProgress (처리 14000건, 실패 3건)
    상태: inProgress (처리 14000건, 실패 3건)
    상태: inProgress (처리 14500건, 실패 3건)
    상태: inProgress (처리 16500건, 실패 3건)
    상태: inProgress (처리 16500건, 실패 3건)
    상태: inProgress (처리 17500건, 실패 3건)
    상태: inProgress (처리 19000건, 실패 3건)
    상태: inProgress (처리 19000건, 실패 3건)
    상태: inProgress (처리 20500건, 실패 4건)
    상태: inProgress (처리 22000건, 실패 4건)
    타임아웃 — 모니터링 종료

✓ 파이프라인 설정 완료!
  Indexer가 매 PT24H 간격으로 자동 실행됩니다.
  신규/변경 데이터가 없으면 자동으로 skip합니다.

  인덱스 목록:
    - const-court-index (semantic: const-semantic)

  수동 실행:
    uv run python scripts/setup_ai_search_pipeline.py --source prec --run
    uv run python scripts/setup_ai_search_p

In [23]:
# 시나리오 B: 캐시 ON 으로 전체 재인덱싱 (1차 - 캐시가 채워짐)
print("="*70, f"\nB. SETUP_ENABLE_CACHE=1  →  전체 재인덱싱 (캐시 채움)  (source={EXP_SOURCE})", "\n"+"="*70)
print(f"시작: {datetime.now().strftime('%H:%M:%S')}")

rc_b, elapsed_b, log_b = run_setup(EXP_SOURCE, cache_on=True)

tail = "\n".join(log_b.splitlines()[-30:])
print(tail)
print(f"\n[B 결과] rc={rc_b}  소요={elapsed_b:.1f}s ({int(elapsed_b//60)}분 {int(elapsed_b%60)}초)")


B. SETUP_ENABLE_CACHE=1  →  전체 재인덱싱 (캐시 채움)  (source=detc) 
시작: 13:12:21
    상태: inProgress (처리 4000건, 실패 0건)
    상태: inProgress (처리 4000건, 실패 0건)
    상태: inProgress (처리 4000건, 실패 0건)
    상태: inProgress (처리 4500건, 실패 0건)
    상태: inProgress (처리 4500건, 실패 0건)
    상태: inProgress (처리 5000건, 실패 0건)
    상태: inProgress (처리 5000건, 실패 0건)
    상태: inProgress (처리 5000건, 실패 0건)
    상태: inProgress (처리 5000건, 실패 0건)
    상태: inProgress (처리 5500건, 실패 1건)
    상태: inProgress (처리 5500건, 실패 1건)
    상태: inProgress (처리 5500건, 실패 1건)
    상태: inProgress (처리 6000건, 실패 1건)
    상태: inProgress (처리 6000건, 실패 1건)
    상태: inProgress (처리 6000건, 실패 1건)
    상태: inProgress (처리 6000건, 실패 1건)
    상태: inProgress (처리 6500건, 실패 2건)
    타임아웃 — 모니터링 종료

✓ 파이프라인 설정 완료!
  Indexer가 매 PT24H 간격으로 자동 실행됩니다.
  신규/변경 데이터가 없으면 자동으로 skip합니다.

  인덱스 목록:
    - const-court-index (semantic: const-semantic)

  수동 실행:
    uv run python scripts/setup_ai_search_pipeline.py --source prec --run
    uv run python scripts/setup_ai_search_pipeline.p

In [ ]:
# 시나리오 B.5: 크롤링으로 신규 파일 추가 → C 에서 incremental update 가 처리할 변경분 생성
# Durable Functions orchestrator (`/api/orchestrators/crawl_preprocess`) 를 직접 호출 → statusQueryGetUri 폴링.
#
# 아키텍처 메모:
# - 정기 운영은 Logic App `logic-crawl-ragi-63325wdo` 가 매일 06:00 KST 동일 orchestrator 호출.
# - 노트북에서는 Logic App 을 거치지 않고 orchestrator HTTP starter 를 직접 호출 → 230s gateway timeout 회피, preprocess 완료까지 정확히 대기.
print("="*70, f"\nB.5 신규 데이터 추가 (Durable orchestrator 직접 호출)", "\n"+"="*70)

ORCH_URL = os.environ.get("AZURE_FUNCTION_CRAWL_ORCH_URL", "")
assert ORCH_URL, "AZURE_FUNCTION_CRAWL_ORCH_URL 가 .env 에 설정되어야 합니다."

prefix_today = f"{EXP_SOURCE}/"
before_total = sum(
    1 for b in container_client.list_blobs(name_starts_with=prefix_today)
    if b.name.endswith(".json")
)
print(f"crawl 전 {EXP_SOURCE}/ 전체 .json blob 수: {before_total:,}")

orch_params = {"source": EXP_SOURCE, "max_pages": "2", "triggered_by": "cache-exp"}
print(f"\norchestrator 호출: {ORCH_URL}  params={orch_params}")

t0 = time.time()
preprocess_docs = 0
preprocess_status = "unknown"
preprocess_files = 0
preprocess_parts = 0
preprocess_errors = 0

start_resp = requests.post(ORCH_URL, params=orch_params, timeout=30)
print(f"  start HTTP {start_resp.status_code}")
start_json = start_resp.json()
status_uri = start_json["statusQueryGetUri"]
print(f"  instanceId={start_json.get('id')}")

# 폴링
poll_interval = 15
max_wait = 1800
last_status = None
output = None
while True:
    elapsed = time.time() - t0
    s = requests.get(status_uri, timeout=30).json()
    rt = s.get("runtimeStatus")
    if rt != last_status:
        print(f"  [{int(elapsed):>4d}s] runtimeStatus={rt}  customStatus={s.get('customStatus')}")
        last_status = rt
    if rt in ("Completed", "Failed", "Terminated"):
        output = s.get("output")
        break
    if elapsed > max_wait:
        print(f"  ⚠️ {max_wait}s 초과 — 폴링 중단")
        break
    time.sleep(poll_interval)

crawl_elapsed = time.time() - t0
print(f"\n  최종상태={last_status}  소요={crawl_elapsed:.1f}s")

# orchestrator output 파싱
# 구조: {"crawl":{src:{saved_files,...}}, "preprocess":{src:{response:{sources:{src:{files,parts,errors,uploaded:[{docs}]}}}}}}
if isinstance(output, dict):
    crawl_info = (output.get("crawl", {}) or {}).get(EXP_SOURCE, {})
    pre_info   = (output.get("preprocess", {}) or {}).get(EXP_SOURCE, {})
    resp = (pre_info.get("response") or {}).get("sources", {}).get(EXP_SOURCE, {}) if isinstance(pre_info.get("response"), dict) else {}
    preprocess_status = pre_info.get("status", last_status)
    preprocess_files = resp.get("files", 0)
    preprocess_parts = resp.get("parts", 0)
    preprocess_errors = len(resp.get("errors", []) or [])
    uploaded = resp.get("uploaded", []) or []
    preprocess_docs = sum(u.get("docs", 0) for u in uploaded)
    saved_files = crawl_info.get("saved_files", 0)
    print(f"  crawl.{EXP_SOURCE}: saved={saved_files}  skipped_existing={crawl_info.get('skipped_existing',0)}")
    print(f"  preprocess.{EXP_SOURCE}: status={preprocess_status}  files={preprocess_files}  "
          f"parts={preprocess_parts}  errors={preprocess_errors}  docs_in_jsonl={preprocess_docs}")

after_total = sum(
    1 for b in container_client.list_blobs(name_starts_with=prefix_today)
    if b.name.endswith(".json")
)
added_n = after_total - before_total
print(f"\ncrawl 후 {EXP_SOURCE}/ 전체 .json blob 수: {after_total:,}  (신규 {added_n:+,}개)")

if added_n <= 0:
    print("  ℹ️ 신규 파일이 추가되지 않았거나 이전 백그라운드 crawl 들과 겹쳐 측정됨.")
    print("     (orchestrator 가 보고한 docs_in_jsonl 로 incremental 입력 확보 여부 판단)")

print(f"\n결과 요약: orchestrator={preprocess_status}  files={preprocess_files}  docs_in_jsonl={preprocess_docs}")
print("→ C (Incremental Update) 진행 가능")


B.5 신규 데이터 추가 (crawl → preprocess fire-and-dispatch) 


crawl 전 detc/ 전체 .json blob 수: 26,356

crawl 호출: https://func-crawl-ragi-63325wdo.azurewebsites.net/api/crawl  params={'source': 'detc', 'max_pages': '2', 'triggered_by': 'cache-exp'}
  HTTP 200  소요=22.6s
  응답 요약: {'status': 'success', 'triggered_by': 'cache-exp', 'date_folder': '2026-05-18', 'counts': {'detc': 0}, 'total_docs': 0, 'total_files': 0, 'total_pre_existing': 26373, 'total_skipped_existing': 0, 'total_upload_failed': 0, 'retried_sources': {'detc': 0}, 'crawl_logs': {'detc': '_logs/2026-05-18/detc/20260518T134942632343.jsonl'}, 'preprocess': {'detc': {'source': 'detc', 'status': 'success', 'http_status': 200, 'attempts': 1, 'elapsed_seconds': 5.58, 'response': {'status': 'success', 'triggered_by': 'crawl-function-after:cache-exp', 'crawl_date': '2026-05-18', 'sources': {'detc': {'files': 1031
  preprocess: {'detc': {'source': 'detc', 'status': 'success', 'http_status': 200, 'attempts': 1, 'elapsed_seconds': 5.58, 'response': {'status': 'success', 'triggered_by': 'crawl-funct

In [29]:
# 시나리오 C: 캐시 ON 상태에서 변경 없이 단순 재실행 (incremental skip 효과)
print("="*70, f"\nC. Incremental run (reset 없음, 변경 없음)  (indexer={EXP_INDEXER})", "\n"+"="*70)
print(f"시작: {datetime.now().strftime('%H:%M:%S')}")

result_c = run_indexer_only(EXP_INDEXER)
elapsed_c = result_c["elapsed_sec"]
print(f"  상태: {result_c['status']}")
print(f"  처리건수(itemCount): {result_c['items_processed']}")
print(f"  실패건수(failedItemCount): {result_c['items_failed']}")
print(f"  시작/종료: {result_c['start_time']} → {result_c['end_time']}")
print(f"\n[C 결과] 소요={elapsed_c:.1f}s")


C. Incremental run (reset 없음, 변경 없음)  (indexer=const-blob-indexer) 
시작: 17:05:28


    [   4s] status=inProgress  startTime=2026-05-18T17:05:32.64Z  items=None  failed=None  (new run)
    [  10s] status=inProgress  startTime=2026-05-18T17:05:32.64Z  items=None  failed=None  (new run)
    [  16s] status=inProgress  startTime=2026-05-18T17:05:32.64Z  items=None  failed=None  (new run)
    [  22s] status=inProgress  startTime=2026-05-18T17:05:32.64Z  items=None  failed=None  (new run)
    [  29s] status=inProgress  startTime=2026-05-18T17:05:32.64Z  items=None  failed=None  (new run)
    [  35s] status=inProgress  startTime=2026-05-18T17:05:32.64Z  items=None  failed=None  (new run)
    [  41s] status=inProgress  startTime=2026-05-18T17:05:32.64Z  items=None  failed=None  (new run)
    [  47s] status=inProgress  startTime=2026-05-18T17:05:32.64Z  items=None  failed=None  (new run)
    [  53s] status=inProgress  startTime=2026-05-18T17:05:32.64Z  items=None  failed=None  (new run)
    [  60s] status=inProgress  startTime=2026-05-18T17:05:32.64Z  items=None  failed=None  

In [ ]:
# 시나리오 D: 캐시 ON 상태에서 다시 한번 reindex (reset + run)
# 기대: blob 은 전부 다시 처리되지만 enrichment cache (특히 임베딩) 가 HIT 되어 B 대비 단축
print("="*70, f"\nD. 캐시 ON Reindex (2차) — 캐시 재사용 검증  (source={EXP_SOURCE})", "\n"+"="*70)
print(f"시작: {datetime.now().strftime('%H:%M:%S')}")

rc_d, elapsed_d, log_d = run_setup(EXP_SOURCE, cache_on=True)

tail = "\n".join(log_d.splitlines()[-30:])
print(tail)
print(f"\n[D 결과] rc={rc_d}  소요={elapsed_d:.1f}s ({int(elapsed_d//60)}분 {int(elapsed_d%60)}초)")

# 비교 안내: B (캐시 ON 1차 reindex) 대비 비교
if "elapsed_b" in globals() and elapsed_b > 0:
    diff = elapsed_b - elapsed_d
    pct = diff / elapsed_b * 100
    print(f"\n  → B(cache-ON 1차 reindex) 대비: {diff:+.1f}s ({pct:+.1f}%) "
          f"{'단축 (임베딩 cache HIT 효과)' if diff > 0 else '단축 없음/오히려 증가'}")

D. 캐시 ON Reindex (2차) — 캐시 재사용 검증  (source=detc) 
시작: 17:12:52
    상태: inProgress (처리 3000건, 실패 0건)
    상태: inProgress (처리 3000건, 실패 0건)
    상태: inProgress (처리 3500건, 실패 0건)
    상태: inProgress (처리 3500건, 실패 0건)
    상태: inProgress (처리 3500건, 실패 0건)
    상태: inProgress (처리 4000건, 실패 0건)
    상태: inProgress (처리 4000건, 실패 0건)
    상태: inProgress (처리 4000건, 실패 0건)
    상태: inProgress (처리 4500건, 실패 0건)
    상태: inProgress (처리 4500건, 실패 0건)
    상태: inProgress (처리 4500건, 실패 0건)
    상태: inProgress (처리 4500건, 실패 0건)
    상태: inProgress (처리 5000건, 실패 0건)
    상태: inProgress (처리 5000건, 실패 0건)
    상태: inProgress (처리 5000건, 실패 0건)
    상태: inProgress (처리 5000건, 실패 0건)
    상태: inProgress (처리 5500건, 실패 1건)
    타임아웃 — 모니터링 종료

✓ 파이프라인 설정 완료!
  Indexer가 매 PT24H 간격으로 자동 실행됩니다.
  신규/변경 데이터가 없으면 자동으로 skip합니다.

  인덱스 목록:
    - const-court-index (semantic: const-semantic)

  수동 실행:
    uv run python scripts/setup_ai_search_pipeline.py --source prec --run
    uv run python scripts/setup_ai_search_pipeline.py --run

[

In [ ]:
# 결과 비교 표 + REPORT_CACHE.md 저장
import pandas as pd
from datetime import datetime

added_for_report = added_n if "added_n" in globals() else 0

rows_cache = [
    {"종류": "Reindex", "시나리오": "A. 캐시 OFF 전체 재인덱싱 (baseline)",
     "소요(초)": round(elapsed_a, 1), "비고": f"rc={rc_a}"},
    {"종류": "Reindex", "시나리오": "B. 캐시 ON 전체 재인덱싱 (1차, 캐시 채움)",
     "소요(초)": round(elapsed_b, 1), "비고": f"rc={rc_b}"},
    {"종류": "Update",  "시나리오": f"C. 캐시 ON Incremental Update (신규 {added_for_report}건)",
     "소요(초)": round(elapsed_c, 1),
     "비고": f"items={result_c.get('items_processed')}, failed={result_c.get('items_failed')}, status={result_c.get('status')}"},
]
if "elapsed_d" in globals():
    rows_cache.append({
        "종류": "Reindex", "시나리오": "D. 캐시 ON 전체 재인덱싱 (2차, 캐시 재사용)",
        "소요(초)": round(elapsed_d, 1), "비고": f"rc={rc_d}",
    })

df_cache = pd.DataFrame(rows_cache)
display(df_cache)

speedup_ca = (elapsed_a / elapsed_c) if elapsed_c > 0 else float("inf")
speedup_cb = (elapsed_b / elapsed_c) if elapsed_c > 0 else float("inf")

print(f"\n=== 핵심 비교 (source={EXP_SOURCE}) ===")
print(f"  A (Reindex, no-cache)        : {elapsed_a:>8.1f}s")
print(f"  B (Reindex, cache 채움)      : {elapsed_b:>8.1f}s   (vs A: {(elapsed_b-elapsed_a):+.1f}s)")
print(f"  C (Update, +{added_for_report}건)              : {elapsed_c:>8.1f}s   (vs A: x{speedup_ca:.1f} 빠름)")
if "elapsed_d" in globals():
    diff_ad = elapsed_a - elapsed_d
    pct_ad  = diff_ad / elapsed_a * 100 if elapsed_a > 0 else 0.0
    print(f"  D (Reindex, cache 재사용)    : {elapsed_d:>8.1f}s   (vs A: {-diff_ad:+.1f}s, {-pct_ad:+.1f}% / 임베딩 cache HIT)")

# REPORT_CACHE.md 작성
md = []
md.append("# Indexer Caching 효과 비교 리포트 (Reindex vs Incremental Update)")
md.append("")
md.append(f"- 생성: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
md.append(f"- 실험 소스: `{EXP_SOURCE}`  /  indexer: `{EXP_INDEXER}`")
md.append(f"- Search: `{SEARCH_ENDPOINT}`")
md.append(f"- B.5 크롤링으로 추가된 신규 파일: **{added_for_report}건**")
md.append("")

# 전체 인덱스별 파일/문서 수 집계
md.append("## 인덱스별 현황")
md.append("")
md.append("| 인덱스 | 한국어명 | Blob 파일수 | 인덱스 문서수 |")
md.append("|--------|----------|------------|-------------|")
_total_blob = 0
_total_docs = 0
for _idx_name, _meta in INDEX_META.items():
    _prefix = _meta['blob_prefix']
    try:
        _blob_n = sum(1 for b in container_client.list_blobs(name_starts_with=_prefix) if b.name.endswith('.json'))
    except Exception:
        _blob_n = 0
    try:
        _doc_r = requests.get(f"{SEARCH_ENDPOINT}/indexes('{_idx_name}')/docs/$count?api-version=2024-07-01", headers=headers, timeout=30)
        _doc_n = int(_doc_r.text.strip()) if _doc_r.status_code == 200 else 0
    except Exception:
        _doc_n = 0
    _total_blob += _blob_n
    _total_docs += _doc_n
    md.append(f"| {_idx_name} | {_meta['name']} | {_blob_n:,} | {_doc_n:,} |")
md.append(f"| **합계** | | **{_total_blob:,}** | **{_total_docs:,}** |")
md.append("")
md.append("## 결과")
md.append("")
md.append(df_cache.to_markdown(index=False))

md.append("")

md.append("## 해석")print("\nREPORT_CACHE.md 작성 완료")

md.append("")    f.write("\n".join(md))

md.append(f"- **C / A 가속비**: x{speedup_ca:.1f}")with open("REPORT_CACHE.md", "w", encoding="utf-8") as f:

md.append(f"- **C / B 가속비**: x{speedup_cb:.1f}")

md.append("- A, B, D 는 `reset` 후 재생성하므로 모두 **전체 재인덱싱(Reindex)** 입니다.")md.append("")

md.append("- C 는 reset 없이 `POST /indexers/{name}/run` 만 호출 → **증분 갱신(Incremental Update)**.")    md.append("- `reset` 은 change tracking 만 무효화하고 enrichment cache 는 보존됩니다. 따라서 D 에서 모든 blob 이 재처리되지만 **임베딩 등 skill 결과는 cache HIT** 으로 재사용되어 시간/비용이 감소합니다.")

md.append("  - change tracking(BLOB LastModified) 로 기존 blob skip")    md.append(f"- 절감: **{diff_ad:+.1f}s ({pct_ad:+.1f}%)**")

md.append("  - SETUP_ENABLE_CACHE=1 의 enrichment cache 로 동일 입력 skill 호출 skip")    md.append(f"- A(no-cache reindex) = **{elapsed_a:.1f}s** vs D(cache 재사용 reindex) = **{elapsed_d:.1f}s**")

if "elapsed_d" in globals():    md.append("")

    diff_ad = elapsed_a - elapsed_d    md.append("### D — Reindex 때도 캐시가 재사용되는가?")

    pct_ad  = diff_ad / elapsed_a * 100 if elapsed_a > 0 else 0.0    md.append("")

,종류,시나리오,소요(초),비고
0,Reindex,A. 캐시 OFF 전체 재인덱싱 (baseline),613.8,rc=0
1,Reindex,"B. 캐시 ON 전체 재인덱싱 (1차, 캐시 채움)",628.7,rc=0
2,Update,C. 캐시 ON Incremental Update (신규 71건),435.4,"items=None, failed=None, status=transientFailure"
3,Reindex,"D. 캐시 ON 전체 재인덱싱 (2차, 캐시 재사용)",615.2,rc=0



=== 핵심 비교 (source=detc) ===
  A (Reindex, no-cache)        :    613.8s
  B (Reindex, cache 채움)      :    628.7s   (vs A: +14.8s)
  C (Update, +71건)              :    435.4s   (vs A: x1.4 빠름)
  D (Reindex, cache 재사용)    :    615.2s   (vs A: +1.4s, +0.2% / 임베딩 cache HIT)

REPORT_CACHE.md 작성 완료


## 7. Knowledge Agent + 4개 Knowledge Source 생성

`04-search-and-query.ipynb` 의 시나리오 D/E/F (agentic retrieval) 가 사용하는 다음을 등록합니다:

- **4개 Knowledge Source** (`ks-prec`, `ks-const`, `ks-interp`, `ks-admin`) ← 위에서 생성한 4개 법령 인덱스에 매핑
- **1개 Knowledge Agent** (`legal-knowledge-agent`) ← `gpt-4o` planner + 4개 source 통합

> 선행 조건: 위 4단계의 인덱싱이 완료되어 4개 인덱스에 문서가 들어있어야 합니다.
> AOAI 호출을 위해 Search MI 에 `Cognitive Services OpenAI User` 권한이 부여되어 있어야 함 (Bicep 에 포함됨).


In [32]:
# Knowledge Sources + Knowledge Agent 생성 (idempotent)
import os, sys, requests as _req
from azure.identity import DefaultAzureCredential
sys.path.insert(0, os.path.abspath('..'))
from src.search.legal_indexes import PREC_INDEX, CONST_INDEX, INTERP_INDEX, ADMIN_INDEX

SEARCH_ENDPOINT = os.environ['AZURE_SEARCH_SERVICE_ENDPOINT']
OPENAI_ENDPOINT = os.environ['AZURE_OPENAI_ENDPOINT']

KS_API_VERSION = "2025-08-01-preview"
PLANNER_DEPLOY = "gpt-4o"   # AOAI deployment for Knowledge Agent planner
AGENT_NAME     = "legal-knowledge-agent"

cred_ks = DefaultAzureCredential(
    exclude_managed_identity_credential=True,
    exclude_workload_identity_credential=True,
)
def _ks_hdr():
    tok = cred_ks.get_token("https://search.azure.com/.default").token
    return {"Authorization": f"Bearer {tok}", "Content-Type": "application/json"}

INDEX_TO_SRC = {
    PREC_INDEX:   ("ks-prec",   "한국 법원 판례",
        "caseName,caseNumber,courtName,courtLevel,judgmentDate,relatedLaws,keywords,holdings,summary"),
    CONST_INDEX:  ("ks-const",  "헌법재판소 결정례",
        "caseName,caseNumber,decisionDate,decisionType,relatedLaws,keywords,holdings,summary"),
    INTERP_INDEX: ("ks-interp", "법제처 법령해석례",
        "title,docNumber,replyDate,interpType,relatedLaws,keywords,querySummary,reply"),
    ADMIN_INDEX:  ("ks-admin",  "행정심판 재결례",
        "caseName,caseNumber,decisionDate,decisionType,committee,relatedLaws,keywords,order,reasonSummary"),
}

# 1) Knowledge Source 4개 (PUT = upsert)
for idx_name, (src_name, desc, fields) in INDEX_TO_SRC.items():
    body = {
        "name": src_name, "kind": "searchIndex", "description": desc,
        "searchIndexParameters": {"searchIndexName": idx_name, "sourceDataSelect": fields},
    }
    url = f"{SEARCH_ENDPOINT}/knowledgesources('{src_name}')?api-version={KS_API_VERSION}"
    r = _req.put(url, headers=_ks_hdr(), json=body)
    print(f"  KS '{src_name}' <- {idx_name}: {r.status_code}")
    if r.status_code >= 400: print("    ", r.text[:300])

# 2) Knowledge Agent (PUT = upsert)
agent_body = {
    "name": AGENT_NAME,
    "description": "Korean legal corpus: 판례 / 헌재 / 법제처 / 행심",
    "models": [{"kind": "azureOpenAI", "azureOpenAIParameters": {
        "resourceUri": OPENAI_ENDPOINT.rstrip("/"),
        "deploymentId": PLANNER_DEPLOY, "modelName": PLANNER_DEPLOY,
    }}],
    "knowledgeSources": [
        {"name": s, "alwaysQuerySource": True, "includeReferences": True,
         "includeReferenceSourceData": True, "maxSubQueries": 4, "rerankerThreshold": 1.5}
        for _, (s, _, _) in INDEX_TO_SRC.items()
    ],
    "outputConfiguration": {"modality": "answerSynthesis", "includeActivity": True, "attemptFastPath": False},
    "requestLimits": {"maxRuntimeInSeconds": 60, "maxOutputSize": 16000},
    "retrievalInstructions": (
        "사용자 질문에 한국 법령·판례 자료가 필요하면 4개 소스에서 hybrid+semantic 검색을 수행한다. "
        "이전 대화 메시지에 사건번호(예: 2019도1234)나 법령명이 언급되어 있으면 그 키워드로 sub-query 를 추가 계획하라."
    ),
}
url = f"{SEARCH_ENDPOINT}/agents('{AGENT_NAME}')?api-version={KS_API_VERSION}"
r = _req.put(url, headers=_ks_hdr(), json=agent_body)
print(f"\nAgent '{AGENT_NAME}': {r.status_code}")
if r.status_code >= 400:
    print(r.text[:500])
else:
    print("✅ Knowledge Agent 등록 완료")


  KS 'ks-prec' <- prec-court-index: 204
  KS 'ks-const' <- const-court-index: 204
  KS 'ks-interp' <- legis-interp-index: 204
  KS 'ks-admin' <- admin-appeal-index: 204

Agent 'legal-knowledge-agent': 204
✅ Knowledge Agent 등록 완료
